# NBHD-GeoJSON

In [1]:
%load_ext autoreload
%autoreload 2
%env ANYWIDGET_HMR=1

env: ANYWIDGET_HMR=1


In [2]:
from shapely import Point, MultiPoint, MultiPolygon
import geopandas as gpd
import numpy as np
import pandas as pd
import geopandas as gpd
from libpysal.cg import alpha_shape
import matplotlib.pyplot as plt
import json
from ipywidgets import Widget
import celldega as dega

/Users/feni/Documents/celldega/dega/lib/python3.12/site-packages/h5py/__init__.py:36: UserWarning: h5py is running against HDF5 1.14.5 when it was built against 1.14.6, this may cause problems
  _warn(("h5py is running against HDF5 {0} when it was built against {1}, "


In [3]:
import spatialdata as sd
from spatialdata_io import xenium

/Users/feni/Documents/celldega/dega/lib/python3.12/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


In [4]:
base_path = 'https://raw.githubusercontent.com/broadinstitute/celldega_Xenium_human_Pancreas_FFPE/main/Landscape_Xenium_V1_human_Pancreas_FFPE_outs_webp/'
zarr_path = 'data/xenium_data/Xenium_V1_human_Pancreas_FFPE_outs.zarr'


In [5]:
path_transformation_matrix = base_path + 'micron_to_image_transform.csv'
transformation_matrix = pd.read_csv(path_transformation_matrix, header=None, sep=" ").values
transformation_matrix

array([[4.705882, 0.      , 0.      ],
       [0.      , 4.705882, 0.      ],
       [0.      , 0.      , 1.      ]])

In [6]:
cluster = pd.read_parquet(base_path + 'cell_clusters/cluster.parquet')

In [7]:
sdata = sd.read_zarr(zarr_path)
sdata

/Users/feni/Documents/celldega/dega/lib/python3.12/site-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
/Users/feni/Documents/celldega/dega/lib/python3.12/site-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
/Users/feni/Documents/celldega/dega/lib/python3.12/site-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
/Users/feni/Documents/celldega/dega/lib/python3.12/site-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
/Users/feni/Documents/celldega/dega/lib/python3.12/site-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, 

SpatialData object, with associated Zarr store: /Users/feni/Documents/celldega/notebooks/data/xenium_data/Xenium_V1_human_Pancreas_FFPE_outs.zarr
├── Images
│     └── 'morphology_focus': DataTree[cyx] (5, 13770, 34155), (5, 6885, 17077), (5, 3442, 8538), (5, 1721, 4269), (5, 860, 2134)
├── Labels
│     ├── 'cell_labels': DataTree[yx] (13770, 34155), (6885, 17077), (3442, 8538), (1721, 4269), (860, 2134)
│     └── 'nucleus_labels': DataTree[yx] (13770, 34155), (6885, 17077), (3442, 8538), (1721, 4269), (860, 2134)
├── Points
│     └── 'transcripts': DataFrame with shape: (<Delayed>, 11) (3D points)
├── Shapes
│     ├── 'cell_boundaries': GeoDataFrame shape: (140702, 1) (2D shapes)
│     ├── 'cell_circles': GeoDataFrame shape: (140702, 2) (2D shapes)
│     └── 'nucleus_boundaries': GeoDataFrame shape: (136531, 1) (2D shapes)
└── Tables
      └── 'table': AnnData (140702, 377)
with coordinate systems:
    ▸ 'global', with elements:
        morphology_focus (Images), cell_labels (Labels), 

In [8]:
adata = sdata.tables["table"]
adata.obs.set_index('cell_id', inplace=True)
adata

AnnData object with n_obs × n_vars = 140702 × 377
    obs: 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'deprecated_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'region', 'z_level', 'nucleus_count', 'cell_labels'
    var: 'gene_ids', 'feature_types', 'genome'
    uns: 'spatialdata_attrs'
    obsm: 'spatial'

In [9]:
adata.obs['cluster'] = cluster

In [10]:
adata.obs.head()

,transcript_counts,control_probe_counts,control_codeword_counts,unassigned_codeword_counts,deprecated_codeword_counts,total_counts,cell_area,nucleus_area,region,z_level,nucleus_count,cell_labels,cluster
cell_id,,,,,,,,,,,,,
aaaadnje-1,37,0,0,0,0,37,44.117658,38.021564,cell_circles,0.0,1.0,1,15
aaacalai-1,60,0,0,0,0,60,66.244221,33.912345,cell_circles,0.0,1.0,2,9
aaacjgil-1,63,0,0,0,0,63,104.491566,52.697346,cell_circles,0.0,1.0,3,15
aaacpcil-1,12,0,0,0,0,12,34.183282,17.520626,cell_circles,0.0,1.0,4,13
aaadhocp-1,143,0,0,0,0,143,149.060787,51.116877,cell_circles,0.0,1.0,5,18


In [11]:
adata.obs

,transcript_counts,control_probe_counts,control_codeword_counts,unassigned_codeword_counts,deprecated_codeword_counts,total_counts,cell_area,nucleus_area,region,z_level,nucleus_count,cell_labels,cluster
cell_id,,,,,,,,,,,,,
aaaadnje-1,37,0,0,0,0,37,44.117658,38.021564,cell_circles,0.0,1.0,1,15
aaacalai-1,60,0,0,0,0,60,66.244221,33.912345,cell_circles,0.0,1.0,2,9
aaacjgil-1,63,0,0,0,0,63,104.491566,52.697346,cell_circles,0.0,1.0,3,15
aaacpcil-1,12,0,0,0,0,12,34.183282,17.520626,cell_circles,0.0,1.0,4,13
aaadhocp-1,143,0,0,0,0,143,149.060787,51.116877,cell_circles,0.0,1.0,5,18
...,...,...,...,...,...,...,...,...,...,...,...,...,...
oiloppgp-1,14,0,0,0,0,14,15.082188,15.082188,cell_circles,6.0,1.0,140698,10
oilpccne-1,2,0,0,0,0,2,5.734844,5.734844,cell_circles,6.0,1.0,140699,6
oimacfoj-1,11,0,0,0,0,11,13.682344,13.682344,cell_circles,6.0,1.0,140700,10


### Load Data

In [12]:
# meta_cell_ini = pd.read_parquet(base_path + 'cell_metadata.parquet')
# cluster = pd.read_parquet(base_path + 'cell_clusters/cluster.parquet')
meta_cluster = pd.read_parquet(base_path + 'cell_clusters/meta_cluster.parquet')
# meta_cell = pd.concat([meta_cell_ini, cluster], axis=1)

In [13]:
meta_cluster.head()

,color,count
1,#1f77b4,17949
2,#ff7f0e,15781
3,#2ca02c,14415
4,#d62728,11840
5,#9467bd,9526


### Calculate Alpha Shape Neighborhoods

In [14]:
path_transformation_matrix

'https://raw.githubusercontent.com/broadinstitute/celldega_Xenium_human_Pancreas_FFPE/main/Landscape_Xenium_V1_human_Pancreas_FFPE_outs_webp/micron_to_image_transform.csv'

In [15]:
meta_cluster.head()

,color,count
1,#1f77b4,17949
2,#ff7f0e,15781
3,#2ca02c,14415
4,#d62728,11840
5,#9467bd,9526


In [16]:
alphas=[20, 50]
gdf_alpha = dega.nbhd.alpha_shape_cell_clusters(
    adata, 
    cat='cluster', 
    alphas=alphas,
    meta_cluster=meta_cluster
)
gdf_alpha = gdf_alpha[gdf_alpha['inv_alpha'] == 50]

In [17]:
gdf_alpha.head()

,name,cat,geometry,inv_alpha,color,area
1_50,1_50,1,"MULTIPOLYGON (((3884.790 66.240, 3843.930 29.8...",50.0,#1f77b4,1.163024e+07
5_50,5_50,5,"MULTIPOLYGON (((775.170 1322.400, 787.630 1374...",50.0,#9467bd,7.615571e+06
4_50,4_50,4,"MULTIPOLYGON (((363.400 2592.870, 409.480 2637...",50.0,#d62728,6.415101e+06
7_50,7_50,7,"MULTIPOLYGON (((171.520 2737.010, 142.620 2718...",50.0,#e377c2,5.761309e+06
2_50,2_50,2,"MULTIPOLYGON (((679.810 1306.350, 693.330 1323...",50.0,#ff7f0e,5.545825e+06


In [20]:
Widget.close_all()
base_url = base_path.rstrip('/')
landscape = dega.viz.Landscape(
    technology='Xenium',
    height=500,
    base_url = base_url,
    nbhd=gdf_alpha,
)
landscape

Landscape(base_url='https://raw.githubusercontent.com/broadinstitute/celldega_Xenium_human_Pancreas_FFPE/main/…

In [19]:
# landscape.nbhd_geojson